In [10]:
# MGPEB — Módulo de Gerenciamento de Pouso e Estabilização de Base
## Missão Aurora Siger — Colônia em Marte

In [11]:
# ============================================
#   MGPEB — Módulo de Gerenciamento de Pouso
#   Missão Aurora Siger
# ============================================

# 1. CADASTRO DOS MÓDULOS EM ESTRUTURAS DE DADOS

# Lista com todos os módulos e seus atributos
modulos = [
    {"nome": "Habitação Principal",   "prioridade": 1, "combustivel": 72, "massa": 18, "criticidade": "Alta",  "chegada": "08:00"},
    {"nome": "Geração de Energia",    "prioridade": 2, "combustivel": 85, "massa": 12, "criticidade": "Alta",  "chegada": "08:45"},
    {"nome": "Laboratório Científico","prioridade": 3, "combustivel": 61, "massa":  9, "criticidade": "Média", "chegada": "09:30"},
    {"nome": "Suporte Médico",        "prioridade": 4, "combustivel": 90, "massa":  7, "criticidade": "Alta",  "chegada": "10:00"},
    {"nome": "Logística e Armazém",   "prioridade": 5, "combustivel": 48, "massa": 22, "criticidade": "Média", "chegada": "10:30"},
    {"nome": "Comunicações",          "prioridade": 6, "combustivel": 55, "massa":  5, "criticidade": "Alta",  "chegada": "11:00"},
    {"nome": "Prod. de Alimentos",    "prioridade": 7, "combustivel": 78, "massa": 11, "criticidade": "Média", "chegada": "11:45"},
    {"nome": "Tratamento de Resíduos","prioridade": 8, "combustivel": 66, "massa":  8, "criticidade": "Baixa", "chegada": "12:30"},
]

# Fila principal (queue) — módulos aguardando autorização
from collections import deque
fila_pouso = deque(modulos)

# Listas auxiliares
pousados  = []   # módulos que pousaram com sucesso
em_espera = []   # módulos com condição irregular
pilha_alertas = []  # pilha LIFO de alertas

print("Módulos cadastrados:", len(fila_pouso))

Módulos cadastrados: 8


In [12]:
# 2. ALGORITMO DE VERIFICAÇÃO COM PORTAS LÓGICAS

def verificar_pouso(modulo, area_disponivel=1, sensores_ok=1, atmosfera_ok=1):
    nome = modulo["nome"]
    comb = modulo["combustivel"]
    crit = modulo["criticidade"]

    # Condições normais (porta AND)
    condicao_normal = (
        comb > 50 and
        area_disponivel == 1 and
        sensores_ok == 1 and
        atmosfera_ok == 1
    )

    # Exceção para módulos críticos (porta AND alternativa)
    excecao_critica = (
        crit == "Alta" and
        comb >= 40 and
        area_disponivel == 1 and
        sensores_ok == 1 and
        atmosfera_ok == 1
    )

    # Decisão final (porta OR)
    decisao_final = condicao_normal or excecao_critica

    # Porta NOT — bloqueio
    bloqueado = not decisao_final

    return decisao_final, bloqueado

print("Função de verificação carregada com sucesso.")

Função de verificação carregada com sucesso.


In [13]:
# 3.1 ALGORITMOS DE BUSCA

def buscar_menor_combustivel(lista):
    return min(lista, key=lambda m: m["combustivel"])

def buscar_maior_prioridade(lista):
    return min(lista, key=lambda m: m["prioridade"])

def buscar_por_criticidade(lista, criticidade):
    return [m for m in lista if m["criticidade"] == criticidade]

# Executando as buscas
print("===== RESULTADOS DE BUSCA =====")
print()

menor_comb = buscar_menor_combustivel(modulos)
print(f"Módulo com menor combustível: {menor_comb['nome']} ({menor_comb['combustivel']}%)")

maior_prior = buscar_maior_prioridade(modulos)
print(f"Módulo com maior prioridade: {maior_prior['nome']} (prioridade {maior_prior['prioridade']})")

criticos = buscar_por_criticidade(modulos, "Alta")
print(f"Módulos de criticidade Alta:")
for m in criticos:
    print(f"  → {m['nome']}")

print()

===== RESULTADOS DE BUSCA =====

Módulo com menor combustível: Logística e Armazém (48%)
Módulo com maior prioridade: Habitação Principal (prioridade 1)
Módulos de criticidade Alta:
  → Habitação Principal
  → Geração de Energia
  → Suporte Médico
  → Comunicações



In [14]:
# 3.2 ALGORITMOS DE ORDENAÇÃO

# Ordenar por prioridade (crescente)
ordenado_prioridade = sorted(modulos, key=lambda m: m["prioridade"])

# Ordenar por combustível (crescente — menos combustível pousa primeiro)
ordenado_combustivel = sorted(modulos, key=lambda m: m["combustivel"])

# Ordenar por massa (decrescente — mais pesado pousa primeiro)
ordenado_massa = sorted(modulos, key=lambda m: m["massa"], reverse=True)

print("===== ORDENAÇÕES DA FILA =====")
print()

print("Por prioridade:")
for m in ordenado_prioridade:
    print(f"  {m['prioridade']}. {m['nome']}")

print()
print("Por menor combustível:")
for m in ordenado_combustivel:
    print(f"  {m['nome']} — {m['combustivel']}%")

print()
print("Por maior massa:")
for m in ordenado_massa:
    print(f"  {m['nome']} — {m['massa']}t")

===== ORDENAÇÕES DA FILA =====

Por prioridade:
  1. Habitação Principal
  2. Geração de Energia
  3. Laboratório Científico
  4. Suporte Médico
  5. Logística e Armazém
  6. Comunicações
  7. Prod. de Alimentos
  8. Tratamento de Resíduos

Por menor combustível:
  Logística e Armazém — 48%
  Comunicações — 55%
  Laboratório Científico — 61%
  Tratamento de Resíduos — 66%
  Habitação Principal — 72%
  Prod. de Alimentos — 78%
  Geração de Energia — 85%
  Suporte Médico — 90%

Por maior massa:
  Logística e Armazém — 22t
  Habitação Principal — 18t
  Geração de Energia — 12t
  Prod. de Alimentos — 11t
  Laboratório Científico — 9t
  Tratamento de Resíduos — 8t
  Suporte Médico — 7t
  Comunicações — 5t


In [15]:
# 3. SIMULAÇÃO DA FILA DE POUSO

print("=" * 55)
print("   MGPEB — SIMULAÇÃO DE POUSO — BASE AURORA SIGER")
print("=" * 55)
print()

while fila_pouso:
    modulo = fila_pouso.popleft()
    autorizado, bloqueado = verificar_pouso(modulo)

    print(f"Módulo: {modulo['nome']}")
    print(f"  Prioridade: {modulo['prioridade']} | Combustível: {modulo['combustivel']}% | Criticidade: {modulo['criticidade']}")

    if autorizado:
        pousados.append(modulo)
        print(f"  RESULTADO: POUSO AUTORIZADO")
    else:
        em_espera.append(modulo)
        alerta = f"ALERTA: {modulo['nome']} bloqueado — combustível insuficiente ({modulo['combustivel']}%)"
        pilha_alertas.append(alerta)
        print(f"  RESULTADO: POUSO BLOQUEADO")

    print()

print("=" * 55)
print(f"Módulos pousados:  {len(pousados)}")
print(f"Módulos em espera: {len(em_espera)}")
print()

if pilha_alertas:
    print("ALERTAS REGISTRADOS:")
    for alerta in reversed(pilha_alertas):
        print(f"  → {alerta}")

print("=" * 55)

   MGPEB — SIMULAÇÃO DE POUSO — BASE AURORA SIGER

Módulo: Habitação Principal
  Prioridade: 1 | Combustível: 72% | Criticidade: Alta
  RESULTADO: POUSO AUTORIZADO

Módulo: Geração de Energia
  Prioridade: 2 | Combustível: 85% | Criticidade: Alta
  RESULTADO: POUSO AUTORIZADO

Módulo: Laboratório Científico
  Prioridade: 3 | Combustível: 61% | Criticidade: Média
  RESULTADO: POUSO AUTORIZADO

Módulo: Suporte Médico
  Prioridade: 4 | Combustível: 90% | Criticidade: Alta
  RESULTADO: POUSO AUTORIZADO

Módulo: Logística e Armazém
  Prioridade: 5 | Combustível: 48% | Criticidade: Média
  RESULTADO: POUSO BLOQUEADO

Módulo: Comunicações
  Prioridade: 6 | Combustível: 55% | Criticidade: Alta
  RESULTADO: POUSO AUTORIZADO

Módulo: Prod. de Alimentos
  Prioridade: 7 | Combustível: 78% | Criticidade: Média
  RESULTADO: POUSO AUTORIZADO

Módulo: Tratamento de Resíduos
  Prioridade: 8 | Combustível: 66% | Criticidade: Baixa
  RESULTADO: POUSO AUTORIZADO

Módulos pousados:  7
Módulos em espera: 1
